In [1]:
import os
from typing import List

import numpy as np
import pandas as pd


def load_compustat_data(path: str) -> pd.DataFrame:
    """
    Load raw Compustat data from a CSV file.
    """
    # Read as text first; numbers will be converted in preprocessing
    df = pd.read_csv(path, dtype=str)
    return df


def preprocess_compustat(df: pd.DataFrame) -> pd.DataFrame:
    """
    Preprocess the Compustat data.
    """
    useful: List[str] = [
        "gvkey", "datadate", "fyear", "conm", "tic",
        "act", "lct", "at", "lt", "seq", "teq",
        "dlc", "dltt", "che", "invt", "wcap",
        "revt", "cogs", "dp", "ebit", "ebitda", "xint",
        "capx", "oancf", "fincf", "ivncf", "wcapch",
        "csho", "mkvalt", "prcc_f", "bkvlps", "nipfc"
    ]

    # Keep useful columns only
    cols_exist = [c for c in useful if c in df.columns]
    df = df[cols_exist]

    # Drop rows missing gvkey or fyear
    if "gvkey" in df.columns:
        df = df.dropna(subset=["gvkey"])
    if "fyear" in df.columns:
        df = df.dropna(subset=["fyear"])

    # Convert datadate to datetime
    if "datadate" in df.columns:
        df["datadate"] = pd.to_datetime(df["datadate"], errors="coerce")

    # Convert numeric columns to float
    numeric_cols = [
        "act", "lct", "at", "lt", "seq", "teq", "dlc", "dltt", "che", "invt", "wcap",
        "revt", "cogs", "dp", "ebit", "ebitda", "xint", "capx", "oancf", "fincf",
        "ivncf", "wcapch", "csho", "mkvalt", "prcc_f", "bkvlps", "nipfc"
    ]
    numeric_cols = [c for c in numeric_cols if c in df.columns]

    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Fill numeric NaNs with 0
    if numeric_cols:
        df[numeric_cols] = df[numeric_cols].fillna(0)

    # Convert fyear to int
    if "fyear" in df.columns:
        df["fyear"] = df["fyear"].astype(int)

    return df


def safe_div(numerator: pd.Series, denominator: pd.Series, default: float = 0.0) -> pd.Series:
    """
    Safe division helper that avoids division by zero and invalid values.
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        res = np.divide(
            numerator,
            denominator,
            out=np.zeros_like(numerator, dtype=float),
            where=denominator != 0,
        )
    res = np.where(np.isfinite(res), res, default)
    return pd.Series(res, index=numerator.index)


def compute_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute multiple financial KPIs from Compustat-style data.
    """
    d = df.copy()

    # Profitability
    if "net_income" in d.columns or "nipfc" in d.columns:
        if "nipfc" in d.columns:
            d["net_income"] = d["nipfc"]

        if "at" in d.columns:
            d["roa"] = safe_div(d["net_income"], d["at"])

        if "teq" in d.columns:
            d["roe"] = safe_div(d["net_income"], d["teq"])

    if "revt" in d.columns and "cogs" in d.columns:
        d["gross_margin"] = safe_div(d["revt"] - d["cogs"], d["revt"])

    if "ebitda" in d.columns and "revt" in d.columns:
        d["ebitda_margin"] = safe_div(d["ebitda"], d["revt"])

    # Leverage
    if "lt" in d.columns and "at" in d.columns:
        d["debt_ratio"] = safe_div(d["lt"], d["at"])

    if "lt" in d.columns and "seq" in d.columns:
        d["debt_to_equity"] = safe_div(d["lt"], d["seq"])

    if {"dltt", "dlc", "seq"}.issubset(d.columns):
        d["total_debt_to_equity"] = safe_div(d["dltt"] + d["dlc"], d["seq"])

    # Liquidity
    if "act" in d.columns and "lct" in d.columns:
        d["current_ratio"] = safe_div(d["act"], d["lct"])

    if {"act", "lct", "invt"}.issubset(d.columns):
        d["quick_ratio"] = safe_div(d["act"] - d["invt"], d["lct"])

    if "che" in d.columns and "lct" in d.columns:
        d["cash_ratio"] = safe_div(d["che"], d["lct"])

    # Coverage
    if "ebit" in d.columns and "xint" in d.columns:
        d["interest_coverage"] = safe_div(d["ebit"], d["xint"].abs())

    # Cash-flow based
    if {"che", "dltt", "dlc"}.issubset(d.columns):
        d["cash_to_debt"] = safe_div(d["che"], d["dltt"] + d["dlc"])

    if {"oancf", "dltt", "dlc"}.issubset(d.columns):
        d["ope_cash_flow_to_debt"] = safe_div(d["oancf"], d["dltt"] + d["dlc"])

    if "oancf" in d.columns and "revt" in d.columns:
        d["cfo_margin"] = safe_div(d["oancf"], d["revt"])

    # Growth (by gvkey, across years)
    if "gvkey" in d.columns and "fyear" in d.columns:
        d = d.sort_values(["gvkey", "fyear"])

        if "revt" in d.columns:
            d["sales_growth"] = d.groupby("gvkey")["revt"].pct_change()

        if "at" in d.columns:
            d["asset_growth"] = d.groupby("gvkey")["at"].pct_change()

        if "seq" in d.columns:
            d["equity_growth"] = d.groupby("gvkey")["seq"].pct_change()

    # Market-based
    if "seq" in d.columns and "mkvalt" in d.columns:
        d["book_to_market"] = safe_div(d["seq"], d["mkvalt"])

    if "prcc_f" in d.columns and "bkvlps" in d.columns:
        d["price_to_book"] = safe_div(d["prcc_f"], d["bkvlps"])

    if "nipfc" in d.columns and "mkvalt" in d.columns:
        d["earnings_yield"] = safe_div(d["nipfc"], d["mkvalt"])

    # Fill remaining NaNs in numeric columns
    kp_cols = [c for c in d.columns if d[c].dtype.kind in "fi"]
    if kp_cols:
        d[kp_cols] = d[kp_cols].fillna(0)

    return d


if __name__ == "__main__":
    path_input = "/files/financial-kpis-analysis-and-distress-prediction/data/raw/compustat_data.csv"
    path_output = "/files/financial-kpis-analysis-and-distress-prediction/data/processed/compustat_kpis.csv"

    os.makedirs(os.path.dirname(path_output), exist_ok=True)

    df_raw = load_compustat_data(path_input)
    df_prep = preprocess_compustat(df_raw)
    df_kpis = compute_kpis(df_prep)

    df_kpis.to_csv(path_output, index=False)
    print(f"KPIs calculated and saved to: {path_output}")


KPIs calculated and saved to: /files/financial-kpis-analysis-and-distress-prediction/data/processed/compustat_kpis.csv
